# Exercice 1 : Un modèle qui exécute du code au chargement

> **Objectif :** voir de ses yeux que charger un modèle au format pickle, c'est exécuter du code choisi par son auteur.
> Puis inspecter de vrais fichiers de Hugging Face **sans jamais les charger**, avec `pickletools`, `fickling` et `modelscan`.

**Le scénario.** Une équipe télécharge un modèle sur Hugging Face et l'ouvre avec `pickle.load` (ou `joblib.load`, ou `torch.load`).
C'est exactement ce qui s'est passé avec les ≈ 100 modèles malveillants trouvés par JFrog en 2024 : le simple chargement ouvrait un *reverse shell*.

**Le plan.**
1. Fabriquer un modèle piégé **inoffensif** (sa charge utile est un simple `print`).
2. Le charger, comme le ferait la victime.
3. Lire le fichier sans l'exécuter (`pickletools`).
4. Télécharger 4 vrais fichiers de Hugging Face et les analyser **statiquement**, dans un processus Python isolé (`python -I`).

## Étape 0 : exécutez cette cellule en premier
Elle liste les dépendances. Sur Colab (notebook ouvert depuis Drive), elle télécharge `helper.py` et les données. Elle doit afficher **Prêt : True**.

In [1]:
# Dépendances : scikit-learn pandas huggingface_hub fickling modelscan
# Récupère helper.py et les données depuis GitHub s'ils ne sont pas à côté du notebook
# (cas d'un notebook ouvert depuis Google Drive). En local, cette cellule ne fait rien.
import os, sys
if not os.path.exists('helper.py'):
    !test -d /content/cours || git clone -q --depth 1 https://github.com/elyesmanai/8IAR805---Cybersecurity-AI.git /content/cours
    %cd "/content/cours/Chapitre 6 - Chaîne d'approvisionnement des modèles"
    !pip -q install fickling modelscan
print('Prêt :', os.path.exists('helper.py'))

Prêt : True


## Étape 1 : Fabriquer un modèle piégé (inoffensif)
**Ce qu'on fait :** on entraîne l'arbre de décision du Ch. 1 (détection de maliciels), puis on le range dans un dictionnaire avec un objet piégé.
La méthode `__reduce__` dit à pickle : « pour recréer cet objet, appelle cette fonction avec ces arguments ». L'attaquant y met ce qu'il veut.

In [2]:
import pickle
from helper import charger_maliciels
from sklearn.tree import DecisionTreeClassifier

X_train, X_test, y_train, y_test = charger_maliciels()
arbre = DecisionTreeClassifier(max_depth=8, random_state=0).fit(X_train, y_train)

class Piege:
    def __reduce__(self):
        # Au chargement, pickle appellera print(...). Un attaquant mettrait os.system, exec, un reverse shell...
        return (print, ("[CHARGE UTILE] Ce texte est affiché par le FICHIER. J'aurais pu lancer n'importe quelle commande.",))

paquet = {'modele': arbre, 'version': '1.0', 'auteur': 'equipe-ml', 'meta': Piege()}
with open('data/modele_piege.pkl', 'wb') as f:
    pickle.dump(paquet, f)
print('Fichier écrit :', os.path.getsize('data/modele_piege.pkl'), 'octets')

Fichier écrit : 11377 octets


**👀 Ce qu'on doit voir :** rien de spécial. La sauvegarde n'exécute rien ; le piège attend le chargement.

## Étape 2 : La victime charge le modèle
**Ce qu'on fait :** on charge le fichier comme le ferait n'importe quel collègue. C'est **le seul** pickle piégé qu'on charge dans ce cours, parce qu'on l'a fabriqué nous-mêmes.

In [3]:
with open('data/modele_piege.pkl', 'rb') as f:
    recu = pickle.load(f)

print('Le modèle fonctionne normalement, accuracy =', round(recu['modele'].score(X_test, y_test), 4))

[CHARGE UTILE] Ce texte est affiché par le FICHIER. J'aurais pu lancer n'importe quelle commande.
Le modèle fonctionne normalement, accuracy = 0.9875


**👀 Ce qu'on doit voir :** le message `[CHARGE UTILE]` s'affiche **avant** même qu'on utilise le modèle, et le modèle marche parfaitement.
La victime n'a aucune raison de se méfier. C'est l'idée de *Sleepy Pickle* (Trail of Bits, 2024) : la charge utile peut même modifier les poids au passage.

## Étape 3 : Lire le fichier sans l'exécuter
**Ce qu'on fait :** un pickle est un petit programme pour une machine à pile. `pickletools.dis` affiche ses instructions **sans les exécuter**.
On regarde d'abord l'objet piégé seul, puis la liste des fonctions que le fichier complet demande d'importer.

In [4]:
import pickletools
pickletools.dis(pickle.dumps(Piege(), protocol=4))

    0: \x80 PROTO      4
    2: \x95 FRAME      127
   11: \x8c SHORT_BINUNICODE 'builtins'
   21: \x94 MEMOIZE    (as 0)
   22: \x8c SHORT_BINUNICODE 'print'
   29: \x94 MEMOIZE    (as 1)
   30: \x93 STACK_GLOBAL
   31: \x94 MEMOIZE    (as 2)
   32: \x8c SHORT_BINUNICODE "[CHARGE UTILE] Ce texte est affiché par le FICHIER. J'aurais pu lancer n'importe quelle commande."
  132: \x94 MEMOIZE    (as 3)
  133: \x85 TUPLE1
  134: \x94 MEMOIZE    (as 4)
  135: R    REDUCE
  136: \x94 MEMOIZE    (as 5)
  137: .    STOP
highest protocol among opcodes = 4


**👀 Ce qu'on doit voir :** `builtins` puis `print` (`STACK_GLOBAL` = « importe cette fonction »), le texte, puis `REDUCE` = « appelle-la ».
`GLOBAL`/`STACK_GLOBAL` suivi de `REDUCE`, c'est la signature de l'exécution de code.

In [5]:
from helper import analyser_isole
analyser_isole('imports', 'data/modele_piege.pkl')['imports']

['builtins.print',
 'numpy.core.multiarray._reconstruct',
 'numpy.core.multiarray.scalar',
 'numpy.dtype',
 'numpy.ndarray',
 'sklearn.tree._classes.DecisionTreeClassifier',
 'sklearn.tree._tree.Tree']

**👀 Ce qu'on doit voir :** six imports `numpy` et `sklearn` (normaux : il faut bien recréer l'arbre) et, en tête de liste, `builtins.print`.
Dans un vrai modèle, la ligne dangereuse se cache parmi les imports légitimes.

**❓ Question pour la classe :** pourquoi ne pas simplement interdire tous les imports ? *Réponse : un modèle scikit-learn ou PyTorch en a besoin pour se reconstruire. On ne peut que filtrer, avec une liste qui sera toujours incomplète.*

## Étape 4 : Télécharger de vrais fichiers depuis Hugging Face
**Ce qu'on fait :** on télécharge 4 fichiers réels dans `data/non_fiable/`. **Télécharger n'exécute rien** ; le danger commence au chargement, et on ne chargera aucun de ces fichiers.
- deux modèles sains : un petit BERT au format PyTorch et un pipeline scikit-learn ;
- deux fichiers de test publiés par un ingénieur de Hugging Face (`mcpotato/42-eicar-street`) pour tester ses propres scanners.

In [6]:
import pandas as pd
from helper import telecharger_hf
fichiers = telecharger_hf()
pd.DataFrame(fichiers)[['depot', 'fichier', 'description']]

,depot,fichier,description
0,hf-internal-testing/tiny-random-bert,pytorch_model.bin,"Petit BERT (format PyTorch, sain)"
1,scikit-learn/tabular-playground,model.pkl,"Pipeline scikit-learn (pickle, sain)"
2,mcpotato/42-eicar-street,danger.dat,Fichier de test de HF : pickle qui exécute du ...
3,mcpotato/42-eicar-street,model_broken_X.pkl,Fichier de test de HF : pickle volontairement ...


## Étape 5 : Que dit Hugging Face de ces dépôts ?
**Ce qu'on fait :** Hugging Face analyse chaque fichier déposé (picklescan, antivirus) et publie son verdict. On l'interroge par son API, sans rien télécharger.
On ajoute `ykilcher/totally-harmless-model`, un modèle de démonstration de 265 Mo qu'on ne télécharge pas.

In [7]:
from helper import verdict_hf
verdict_hf(['hf-internal-testing/tiny-random-bert', 'scikit-learn/tabular-playground',
            'mcpotato/42-eicar-street', 'ykilcher/totally-harmless-model'])

,dépôt,analyse faite,fichiers signalés
0,hf-internal-testing/tiny-random-bert,True,aucun
1,scikit-learn/tabular-playground,True,aucun
2,mcpotato/42-eicar-street,True,"model_broken_X.pkl (unsafe), danger.dat (unsaf..."
3,ykilcher/totally-harmless-model,True,pytorch_model.bin (unsafe)


**👀 Ce qu'on doit voir :** les deux dépôts sains sont propres ; `danger.dat` et `model_broken_X.pkl` sont marqués `unsafe` ; le modèle « totally harmless » aussi.
Attention : Hugging Face **signale** mais **n'empêche pas** le téléchargement (constat de JFrog, 2024).

## Étape 6 : Analyse statique isolée (1/3) : en-tête et imports
**Ce qu'on fait :** chaque lecture d'un fichier non fiable passe par `analyser_isole(...)`, qui lance `python -I helper.py ...` dans un **processus séparé**.
Le mode `-I` empêche Python d'importer du code depuis le dossier courant (un `json.py` piégé à côté du modèle ne serait pas chargé).

In [8]:
lignes = []
for f in fichiers:
    entete = analyser_isole('entete', f['chemin'])
    imp = analyser_isole('imports', f['chemin'])
    lignes.append({'fichier': f['fichier'], 'format': entete['nature'], 'octets': entete['taille'],
                   'imports demandés': ', '.join(imp['imports']), 'lecture': imp['erreur'] or 'complète'})
pd.set_option('display.max_colwidth', 120)
pd.DataFrame(lignes)

,fichier,format,octets,imports demandés,lecture
0,pytorch_model.bin,ZIP (format PyTorch),540217,"collections.OrderedDict, torch.FloatStorage, torch.LongStorage, torch._utils._rebuild_tensor_v2",complète
1,model.pkl,pickle (protocole 3),6824,"builtins.slice, numpy.core.multiarray._reconstruct, numpy.core.multiarray.scalar, numpy.dtype, numpy.float64, numpy....",complète
2,danger.dat,pickle (protocole 4),66,builtins.eval,complète
3,model_broken_X.pkl,pickle (protocole 2),89,builtins.exec,ValueError: not enough data in stream to read uint4


**👀 Ce qu'on doit voir :**
- le BERT est une archive **ZIP** qui contient un pickle (`data.pkl`) : il importe seulement des fonctions `torch` de reconstruction ;
- le pipeline scikit-learn importe des classes `sklearn` et `numpy` ;
- `danger.dat` importe `builtins.eval` : exécution de code ;
- `model_broken_X.pkl` importe `builtins.exec`, **et sa lecture plante** à la fin. Le fichier est volontairement cassé.

## Étape 7 : Analyse statique isolée (2/3) : fickling
**Ce qu'on fait :** `fickling` (Trail of Bits) **décompile** le pickle en code Python lisible et donne un verdict.

In [9]:
lignes = []
for f in fichiers:
    r = analyser_isole('fickling', f['chemin'])
    lignes.append({'fichier': f['fichier'], 'verdict fickling': r['verdict'], 'première alerte': (r['alertes'] or [''])[0][:90]})
pd.DataFrame(lignes)

,fichier,verdict fickling,première alerte
0,pytorch_model.bin,LIKELY_UNSAFE,Call to `_rebuild_tensor_v2(...)` can execute arbitrary code and is inherently unsafe
1,model.pkl,LIKELY_UNSAFE,Call to `ColumnTransformer.__new__(...)` can execute arbitrary code and is inherently unsa
2,danger.dat,OVERTLY_MALICIOUS,Call to `eval(...)` is almost certainly evidence of a malicious pickle file
3,model_broken_X.pkl,OVERTLY_MALICIOUS,Call to `exec(...)` is almost certainly evidence of a malicious pickle file


In [10]:
for nom in ['danger.dat', 'model_broken_X.pkl']:
    chemin = next(f['chemin'] for f in fichiers if f['fichier'] == nom)
    print(f'===== {nom} décompilé =====')
    print(analyser_isole('fickling', chemin)['code_decompile'], '\n')

===== danger.dat décompilé =====


from builtins import eval
_var0 = eval("exec('''print('hack3d')''')")
result = _var0 

===== model_broken_X.pkl décompilé =====


from builtins import exec
_var0 = exec("\nf = open('my_file.txt', 'a'); f.write('Malicious'); f.close()") 



**👀 Ce qu'on doit voir :** le code caché apparaît en clair : `eval(... print('hack3d') ...)` et `exec(... open('my_file.txt', 'a') ...)`.
fickling classe les deux fichiers piégés `OVERTLY_MALICIOUS`, et signale que `model_broken_X.pkl` a des opcodes invalides.
Mais il classe aussi les deux modèles **sains** `LIKELY_UNSAFE` : pour lui, tout import hors bibliothèque standard peut exécuter du code. Il a raison… et ce n'est pas exploitable au quotidien.

## Étape 8 : Analyse statique isolée (3/3) : modelscan
**Ce qu'on fait :** `modelscan` (Protect AI) fonctionne comme un antivirus : il cherche des opérateurs dangereux connus dans une liste.

In [11]:
lignes = []
for f in fichiers:
    r = analyser_isole('modelscan', f['chemin'])
    lignes.append({'fichier': f['fichier'],
                   'problèmes trouvés': ', '.join(f"{p['gravite']} {p['operateur']}" for p in r.get('problemes', [])) or 'aucun',
                   'erreurs de lecture': ' | '.join(r.get('erreurs', [])) or '-'})
pd.DataFrame(lignes)

,fichier,problèmes trouvés,erreurs de lecture
0,pytorch_model.bin,aucun,-
1,model.pkl,aucun,-
2,danger.dat,CRITICAL builtins.eval,-
3,model_broken_X.pkl,aucun,Parsing error: not enough data in stream to read uint4


**👀 Ce qu'on doit voir :** modelscan trouve `CRITICAL builtins.eval` dans `danger.dat`. Mais sur `model_broken_X.pkl`, il annonce **« aucun problème »** et une simple erreur de lecture.
C'est exactement la technique **nullifAI** (ReversingLabs, février 2025) : la charge utile est **au début** du flux, le fichier est cassé **à la fin**. `pickle.load` exécute la charge utile avant de planter ; le scanner, lui, abandonne sur l'erreur.

**❓ Question pour la classe :** une erreur de lecture dans un scanner, c'est « pas de problème » ou « problème » ? *Réponse : un fichier illisible par le scanner doit être bloqué, jamais accepté.*

## Ce qu'il faut retenir
- **Charger un pickle = exécuter du code.** `pickle.load`, `joblib.load` et `torch.load` (sans `weights_only=True`) sont concernés. Le modèle peut fonctionner parfaitement et avoir quand même exécuté une charge utile.
- L'analyse **statique** (pickletools, fickling, modelscan) se fait sans charger, dans un processus isolé, sur des fichiers rangés à part.
- Les scanners sont des listes noires : ils ratent ce qu'ils ne connaissent pas, et un fichier **cassé** peut les aveugler (nullifAI). Un fichier que le scanner ne sait pas lire doit être **rejeté**.
- La vraie défense n'est pas de mieux scanner le pickle, c'est de **ne plus utiliser de format qui contient du code**.

**Exercice suivant :** vérifier qu'un modèle n'a pas été modifié (empreinte), puis le convertir dans un format qui ne contient que des données.